In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import pickle
import os
import re
import warnings

warnings.filterwarnings("ignore")


data = pd.read_csv("group1_interpolated_complete.csv")
adf_results = pd.read_csv("adf_results.csv")
data['Month'] = pd.to_datetime(data['Month'])

stations = data['Stations'].unique()
parameters = data.columns.difference(['Month', 'Stations'])

#Create folders
os.makedirs("saved_models", exist_ok=True)


def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)

def get_d_value(station, parameter):
    match = adf_results[(adf_results['Station'] == station) &
                        (adf_results['Parameter'] == parameter)]
    if match.empty:
        return 1
    return 0 if match.iloc[0]['p-value'] < 0.05 else 1


all_metrics = []
best_model_records = []

for station in stations:
    station_data = data[data['Stations'] == station].copy()
    station_data.set_index('Month', inplace=True)

    for param in parameters:
        ts = station_data[param].dropna()
        if len(ts) < 20:
            continue

        d = get_d_value(station, param)

        #80:20 split
        split_idx = int(len(ts) * 0.8)
        train_ts = ts.iloc[:split_idx]
        val_ts = ts.iloc[split_idx:]

        best_rmse = float("inf")
        best_order = None

        for p in range(4):
            for q in range(4):
                try:
                    model = ARIMA(train_ts, order=(p, d, q))
                    model_fit = model.fit()

                    forecast = model_fit.forecast(steps=len(val_ts))
                    forecast.index = val_ts.index

                    valid = val_ts.replace(0, np.nan).dropna()
                    forecast = forecast.loc[valid.index]

                    mae = mean_absolute_error(valid, forecast)
                    mape = mean_absolute_percentage_error(valid, forecast) * 100
                    rmse = np.sqrt(mean_squared_error(valid, forecast))

                    # Save model to file
                    filename = f"saved_models/{clean_filename(station)}_{clean_filename(param)}_p{p}_d{d}_q{q}.pkl"
                    with open(filename, 'wb') as f:
                        pickle.dump(model_fit, f)

                    # Save metrics
                    all_metrics.append({
                        "Station": station,
                        "Parameter": param,
                        "p": p,
                        "d": d,
                        "q": q,
                        "MAE": mae,
                        "MAPE": mape,
                        "RMSE": rmse
                    })

                    #Track best
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_order = (p, d, q)

                except Exception as e:
                    all_metrics.append({
                        "Station": station,
                        "Parameter": param,
                        "p": p,
                        "d": d,
                        "q": q,
                        "MAE": None,
                        "MAPE": None,
                        "RMSE": None,
                        "Error": str(e)
                    })

        # Save best model config
        if best_order:
            best_model_records.append({
                "Station": station,
                "Parameter": param,
                "Best_p": best_order[0],
                "Best_d": best_order[1],
                "Best_q": best_order[2],
                "Best_RMSE": best_rmse
            })

pd.DataFrame(all_metrics).to_csv("arima_gridsearch_metrics_group_a.csv", index=False)
pd.DataFrame(best_model_records).to_csv("best_arima_group_a.csv", index=False)

print("🎯 Done: All models, metrics, and best_model.csv saved.")

In [ ]:

import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")

best_models_df = pd.read_csv("best_model.csv")
data = pd.read_csv("group1_interpolated_complete.csv")
data['Month'] = pd.to_datetime(data['Month'])
data.set_index('Month', inplace=True)

os.makedirs("group1_forecast_plots", exist_ok=True)
os.makedirs("group1_residual_plots", exist_ok=True)

def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)


group1_stations = data['Stations'].unique()

for _, row in best_models_df.iterrows():
    station = row['Station']
    parameter = row['Parameter']
    p, d, q = int(row['Best_p']), int(row['Best_d']), int(row['Best_q'])

    if station not in group1_stations:
        continue

    try:
        #Load pickled model
        model_path = f"saved_models/{clean_filename(station)}_{clean_filename(parameter)}_p{p}_d{d}_q{q}.pkl"
        with open(model_path, 'rb') as f:
            model_fit = pickle.load(f)

        
        ts = data[data['Stations'] == station][parameter].dropna()
        if len(ts) < 20:
            continue

        split_idx = int(len(ts) * 0.8)
        train_ts = ts.iloc[:split_idx]
        test_ts = ts.iloc[split_idx:]


        forecast = model_fit.forecast(steps=len(test_ts))
        forecast.index = test_ts.index

        plt.figure(figsize=(12, 5))
        plt.plot(ts, label='Actual', color='blue')
        plt.plot(forecast, label='Forecast', color='red')
        plt.title(f"{parameter} at {station} - ARIMA({p},{d},{q}) Forecast")
        plt.xlabel("Date")
        plt.ylabel(parameter)
        plt.legend()
        plt.tight_layout()
        forecast_path = f"group1_forecast_plots/{clean_filename(station)}_{clean_filename(parameter)}_forecast.png"
        plt.savefig(forecast_path)
        plt.close()

        residuals = model_fit.resid
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].plot(residuals, color='blue')
        axes[0].set_title(f"Residuals - {parameter} at {station}")
        axes[0].set_xlabel("Time")
        axes[0].set_ylabel("Residual")

        axes[1].hist(residuals, bins=30, edgecolor='black', color='gray', density=True)
        axes[1].set_title(f"Residual Distribution - {parameter}")
        axes[1].set_xlabel("Residual Value")
        axes[1].set_ylabel("Density")

        plt.tight_layout()
        residual_path = f"group1_residual_plots/{clean_filename(station)}_{clean_filename(parameter)}_residuals.png"
        plt.savefig(residual_path)
        plt.close()

        print(f"✅ Saved plots for {station} - {parameter}")

    except Exception as e:
        print(f"⚠️ Could not process {station} - {parameter}: {e}")

✅ Saved plots for Stn. I (Central West Bay) - Ammonia (mg/L)
✅ Saved plots for Stn. I (Central West Bay) - BOD (mg/L)
✅ Saved plots for Stn. I (Central West Bay) - Dissolved Oxygen (mg/L)
✅ Saved plots for Stn. I (Central West Bay) - Fecal Coliform, MPN/100ml (Geomean)
✅ Saved plots for Stn. I (Central West Bay) - Inorganic Phospate (mg/L)
✅ Saved plots for Stn. I (Central West Bay) - Nitrate (mg/L)
✅ Saved plots for Stn. I (Central West Bay) - pH (units)
✅ Saved plots for Stn. II (East Bay) - Ammonia (mg/L)
✅ Saved plots for Stn. II (East Bay) - BOD (mg/L)
✅ Saved plots for Stn. II (East Bay) - Dissolved Oxygen (mg/L)
✅ Saved plots for Stn. II (East Bay) - Fecal Coliform, MPN/100ml (Geomean)
✅ Saved plots for Stn. II (East Bay) - Inorganic Phospate (mg/L)
✅ Saved plots for Stn. II (East Bay) - Nitrate (mg/L)
✅ Saved plots for Stn. II (East Bay) - pH (units)
✅ Saved plots for Stn. IV (Central Bay) - Ammonia (mg/L)
✅ Saved plots for Stn. IV (Central Bay) - BOD (mg/L)
✅ Saved plots for S

In [3]:
import os
import pandas as pd
import re


best_models_df = pd.read_csv("best_model_group2.csv")

model_dir = "saved_models"


def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)


best_model_filenames = set()

for _, row in best_models_df.iterrows():
    station = row["Station"]
    parameter = row["Parameter"]
    p, d, q = int(row["Best_p"]), int(row["Best_d"]), int(row["Best_q"])

    filename = f"{clean_filename(station)}_{clean_filename(parameter)}_p{p}_d{d}_q{q}.pkl"
    best_model_filenames.add(filename)


for file in os.listdir(model_dir):
    if file.endswith(".pkl") and file not in best_model_filenames:
        path = os.path.join(model_dir, file)
        os.remove(path)
        print(f"🗑️ Deleted: {file}")

print("✅ Cleanup complete. Only best models remain.")

🗑️ Deleted: Stn__XX__GEMS__BOD__mg_L__p2_d2_q0.pkl
🗑️ Deleted: Stn__XXI__Cardona__Ammonia__mg_L__p1_d0_q1.pkl
🗑️ Deleted: Stn__XXII__Jala_jala__BOD__mg_L__p2_d1_q2.pkl
🗑️ Deleted: Stn__XXIII__Lumban__Dissolved_Oxygen__mg_L__p2_d0_q3.pkl
🗑️ Deleted: Stn__XX__GEMS__Dissolved_Oxygen__mg_L__p1_d0_q3.pkl
🗑️ Deleted: Stn__XXI__Cardona__BOD__mg_L__p2_d2_q2.pkl
🗑️ Deleted: Stn__XIX__Muntinlupa__Inorganic_Phospate__mg_L__p3_d1_q2.pkl
🗑️ Deleted: Stn__XIX__Muntinlupa__pH__units__p1_d2_q1.pkl
🗑️ Deleted: Stn__XXII__Jala_jala__pH__units__p3_d1_q3.pkl
🗑️ Deleted: Stn__XX__GEMS__pH__units__p3_d2_q1.pkl
🗑️ Deleted: Stn__XX__GEMS__Nitrate__mg_L__p3_d1_q0.pkl
🗑️ Deleted: Stn__XIX__Muntinlupa__Ammonia__mg_L__p2_d0_q0.pkl
🗑️ Deleted: Stn__XIII__Taytay__Inorganic_Phospate__mg_L__p3_d1_q2.pkl
🗑️ Deleted: Stn__XXII__Jala_jala__Ammonia__mg_L__p2_d0_q0.pkl
🗑️ Deleted: Stn__XXII__Jala_jala__Ammonia__mg_L__p2_d0_q1.pkl
🗑️ Deleted: Stn__XIX__Muntinlupa__Ammonia__mg_L__p2_d0_q1.pkl
🗑️ Deleted: Stn__XX__GEMS__Nitr

In [ ]:
# for Group 2

warnings.filterwarnings("ignore")

data = pd.read_csv("group2_interpolated_complete.csv")
adf_results = pd.read_csv("adf_results.csv")
best_model_csv_path = "best_model.csv"
existing_best_models = pd.read_csv(best_model_csv_path)

data['Month'] = pd.to_datetime(data['Month'])
stations = data['Stations'].unique()
parameters = data.columns.difference(['Month', 'Stations'])


os.makedirs("saved_models", exist_ok=True)

def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)

def get_d_value(station, parameter):
    match = adf_results[(adf_results['Station'] == station) &
                        (adf_results['Parameter'] == parameter)]
    if match.empty:
        return 1
    return 0 if match.iloc[0]['p-value'] < 0.05 else 1

all_metrics = []
best_model_records = []

for station in stations:
    station_data = data[data['Stations'] == station].copy()
    station_data.set_index('Month', inplace=True)

    for param in parameters:
        ts = station_data[param].dropna()
        if len(ts) < 20:
            continue

        d = get_d_value(station, param)
        split_idx = int(len(ts) * 0.8)
        train_ts = ts.iloc[:split_idx]
        val_ts = ts.iloc[split_idx:]

        best_rmse = float("inf")
        best_order = None

        for p in range(4):
            for q in range(4):
                try:
                    model = ARIMA(train_ts, order=(p, d, q))
                    model_fit = model.fit()

                    forecast = model_fit.forecast(steps=len(val_ts))
                    forecast.index = val_ts.index

                    valid = val_ts.replace(0, np.nan).dropna()
                    forecast = forecast.loc[valid.index]

                    mae = mean_absolute_error(valid, forecast)
                    mape = mean_absolute_percentage_error(valid, forecast) * 100
                    rmse = np.sqrt(mean_squared_error(valid, forecast))

                    model_filename = f"{clean_filename(station)}_{clean_filename(param)}_p{p}_d{d}_q{q}.pkl"
                    with open(f"saved_models/{model_filename}", 'wb') as f:
                        pickle.dump(model_fit, f)

                    all_metrics.append({
                        "Station": station,
                        "Parameter": param,
                        "p": p,
                        "d": d,
                        "q": q,
                        "MAE": mae,
                        "MAPE": mape,
                        "RMSE": rmse
                    })

                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_order = (p, d, q)

                except Exception as e:
                    all_metrics.append({
                        "Station": station,
                        "Parameter": param,
                        "p": p,
                        "d": d,
                        "q": q,
                        "MAE": None,
                        "MAPE": None,
                        "RMSE": None,
                        "Error": str(e)
                    })

        if best_order:
            best_model_records.append({
                "Station": station,
                "Parameter": param,
                "Best_p": best_order[0],
                "Best_d": best_order[1],
                "Best_q": best_order[2],
                "Best_RMSE": best_rmse
            })

pd.DataFrame(all_metrics).to_csv("group2_arima_gridsearch_metrics.csv", index=False)

#Append new best models to existing file
updated_best_models = pd.concat([existing_best_models, pd.DataFrame(best_model_records)], ignore_index=True)
updated_best_models.to_csv("best_model.csv", index=False)

print("✅ All models trained, metrics saved, and best_model.csv updated.")

✅ All models trained, metrics saved, and best_model.csv updated.


In [ ]:
warnings.filterwarnings("ignore")

data = pd.read_csv("group2_interpolated_complete.csv")
data['Month'] = pd.to_datetime(data['Month'])
data.set_index('Month', inplace=True)

best_models_df = pd.read_csv("best_model.csv")

os.makedirs("group2_forecast_plots", exist_ok=True)
os.makedirs("group2_residual_plots", exist_ok=True)

def clean_filename(text):
    return re.sub(r'[^a-zA-Z0-9_]', '_', text)

group2_stations = data['Stations'].unique()

for _, row in best_models_df.iterrows():
    station = row["Station"]
    parameter = row["Parameter"]
    p, d, q = int(row["Best_p"]), int(row["Best_d"]), int(row["Best_q"])

    if station not in group2_stations:
        continue

    try:
        ts = data[data['Stations'] == station][parameter].dropna()
        if len(ts) < 20:
            continue

        #Load saved model
        model_path = f"saved_models/{clean_filename(station)}_{clean_filename(parameter)}_p{p}_d{d}_q{q}.pkl"
        with open(model_path, 'rb') as f:
            model_fit = pickle.load(f)

        split_idx = int(len(ts) * 0.8)
        train_ts = ts.iloc[:split_idx]
        test_ts = ts.iloc[split_idx:]

        forecast = model_fit.forecast(steps=len(test_ts))
        forecast.index = test_ts.index

        plt.figure(figsize=(12, 5))
        plt.plot(ts, label="Actual", color="blue")
        plt.plot(forecast, label="Forecast", color="red")
        plt.title(f"{parameter} at {station} - Forecast (ARIMA{(p,d,q)})")
        plt.xlabel("Date")
        plt.ylabel(parameter)
        plt.legend()
        plt.tight_layout()

        forecast_plot_path = f"group2_forecast_plots/{clean_filename(station)}_{clean_filename(parameter)}_forecast.png"
        plt.savefig(forecast_plot_path)
        plt.close()

        residuals = model_fit.resid
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].plot(residuals, color='blue')
        axes[0].set_title(f"Residuals - {parameter} at {station}")
        axes[0].set_xlabel("Time")
        axes[0].set_ylabel("Residual")

        axes[1].hist(residuals, bins=30, color='gray', edgecolor='black', density=True)
        axes[1].set_title(f"Residual Distribution - {parameter}")
        axes[1].set_xlabel("Residual Value")
        axes[1].set_ylabel("Density")

        plt.tight_layout()

        residual_plot_path = f"group2_residual_plots/{clean_filename(station)}_{clean_filename(parameter)}_residuals.png"
        plt.savefig(residual_plot_path)
        plt.close()

        print(f"✅ Plotted & saved for {station} - {parameter}")

    except Exception as e:
        print(f"⚠️ Error with {station} - {parameter}: {e}")

✅ Plotted & saved for Stn. XIII (Taytay) - Ammonia (mg/L)
✅ Plotted & saved for Stn. XIII (Taytay) - BOD (mg/L)
✅ Plotted & saved for Stn. XIII (Taytay) - Dissolved Oxygen (mg/L)
✅ Plotted & saved for Stn. XIII (Taytay) - Fecal Coliform, MPN/100ml (Geomean)
✅ Plotted & saved for Stn. XIII (Taytay) - Inorganic Phospate (mg/L)
✅ Plotted & saved for Stn. XIII (Taytay) - Nitrate (mg/L)
✅ Plotted & saved for Stn. XIII (Taytay) - pH (units)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - Ammonia (mg/L)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - BOD (mg/L)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - Dissolved Oxygen (mg/L)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - Fecal Coliform, MPN/100ml (Geomean)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - Inorganic Phospate (mg/L)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - Nitrate (mg/L)
✅ Plotted & saved for Stn. XIX (Muntinlupa) - pH (units)
✅ Plotted & saved for Stn. XX (GEMS) - Ammonia (mg/L)
✅ Plotted & saved for Stn. XX (GEMS) - BOD (mg/